## Bloomberg Data

In [16]:
from pathlib import Path
import pandas as pd

# The notebook and all input files are in the same folder
base = Path.cwd()

# Input files
raw_file = base / "bfix_raw.xlsx"
wmr_majors_file = base / "wmr_fixings_majors.csv"
wmr_exotics_file = base / "wmr_fixings_exotics.csv"

# Output files
output_majors_file = base / "bfix_fixings_majors.csv"
output_exotics_file = base / "bfix_fixings_exotics.csv"

# Read raw Bloomberg data
raw = pd.read_excel(
    raw_file,
    sheet_name=0,
    engine="openpyxl"
)

raw.columns = raw.columns.astype(str).str.strip()
raw["Timestamp"] = pd.to_datetime(raw["Timestamp"], errors="raise")

# Read the required headers and column order from the WMR files
major_columns = pd.read_csv(
    wmr_majors_file,
    sep=";",
    nrows=0
).columns.tolist()[1:]

exotic_columns = pd.read_csv(
    wmr_exotics_file,
    sep=";",
    nrows=0
).columns.tolist()[1:]


def reshape_bloomberg(raw_data, wmr_columns):
    # Convert headers such as EURUSDFIXMP=WM into EURUSD
    pairs = [
        column.replace("FIXMP=WM", "")
        for column in wmr_columns
    ]

    hourly_data = []

    for hour in (10, 11):
        part = pd.DataFrame()

        part["Timestamp"] = (
            raw_data["Timestamp"].dt.normalize()
            + pd.to_timedelta(hour, unit="h")
        )

        for pair in pairs:
            source_column = f"{pair}_{hour}"

            if source_column in raw_data.columns:
                part[pair] = raw_data[source_column].values
            else:
                # Keeps missing Bloomberg currencies, such as
                # EURBAM and EURMDL, as empty columns
                part[pair] = pd.NA

        hourly_data.append(part)

    result = pd.concat(
        hourly_data,
        ignore_index=True
    )

    result = result.sort_values(
        "Timestamp"
    ).reset_index(drop=True)

    # Use exactly the same column names as the WMR files
    result.columns = ["Timestamp"] + wmr_columns

    return result


# Reshape Bloomberg data
bfix_majors = reshape_bloomberg(
    raw,
    major_columns
)

bfix_exotics = reshape_bloomberg(
    raw,
    exotic_columns
)

# CSV export settings matching the WMR files
csv_settings = {
    "sep": ";",
    "decimal": ",",
    "index": False,
    "date_format": "%Y-%m-%d %H:%M:%S",
    "encoding": "utf-8"
}

# Export both files
bfix_majors.to_csv(
    output_majors_file,
    **csv_settings
)

bfix_exotics.to_csv(
    output_exotics_file,
    **csv_settings
)

print(
    f"Created {output_majors_file.name}: "
    f"{len(bfix_majors)} rows"
)

print(
    f"Created {output_exotics_file.name}: "
    f"{len(bfix_exotics)} rows"
)

Created bfix_fixings_majors.csv: 528 rows
Created bfix_fixings_exotics.csv: 528 rows
